In [269]:
from backend.backend import * 
import argparse
import os
from rich.console import Console
from tnn_mdls.func_mdls import *
from tnn_mdls.tb_func_mdls import *
from tnn_mdls.sim_utils import *
import time
from veriloggen import *
import copy
from model import Model
import numpy as np
import random
from layer import Layer
from utils import *
from tnn import *

In [270]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import save_image
from torch.distributions.bernoulli import Bernoulli
import matplotlib.pyplot as plt

In [271]:
mnist_posneg = MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor(),PosNeg(0.5),transforms.CenterCrop(3)]))
mnist_img = MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor(), transforms.CenterCrop(3)]))

In [272]:
img, label = mnist_posneg[0]

In [5]:
#plt.imshow(img.squeeze(), cmap="gray");

In [4]:
### Random Variable Generation ###

def gen_brv(wave=10, ucapture=256, usearch=16, ubackoff=768, umin=32, wres=3):
    
    wmax        = 2**(wres)-1
    bcapture    = Bernoulli(ucapture/1024)
    bsearch     = Bernoulli(usearch/1024)
    bbackoff    = Bernoulli(ubackoff/1024)
    bmin        = Bernoulli(umin/1024)
    w           = torch.Tensor([float(l) for l in range(wmax+1)])
    bF = Bernoulli((w/wmax)*(1-w/wmax))

    datasize    = wave
    rvcapture   = bcapture.sample([datasize])
    rvsearch    = bsearch.sample([datasize])
    rvbackoff   = bbackoff.sample([datasize])
    rvmin       = bmin.sample([datasize])
    rvF = bF.sample([datasize])

    end         = time.time()

    return rvcapture, rvsearch, rvbackoff, rvmin, rvF

In [8]:
def MNIST_single_column(num_synapse=18, thres=11, tres=1, wres=3, wave=10, verbose=False):
    
    # Generate brv random variables
    rvcapture, rvsearch, rvbackoff, rvmin, rvF = gen_brv(wave=wave, ucapture=512, usearch=512, ubackoff=64, umin=64, wres=wres)
    
    # Hard-coded PyTorch config
    layer = TNNColumnLayer(inputsize=3, rfsize=3, stride=1, nprev=2, neurons=1, theta=thres,\
    timeres=tres-1, wres=wres, ntype="rnl", ramp=1, w_init="zero", k=1, stoch="low", reward_en=0)
    
    layer_output = []
    layer_weights_out = []
    
    f = open("tnn_mdls/tests/MNIST_single_column", "w")
    
    for w in range(wave):
        
        f.write('# Wave: '+ str(w) +'\n')
        img, label = mnist_posneg[w]
        
        output_spiketimes, input_spiketimes, li_spiketimes = layer(img)
        layer.weights = layer.stdp(input_spiketimes, output_spiketimes, layer.weights,\
                         rvcapture[w], rvsearch[w], rvbackoff[w], rvmin[w], rvF[w])
        
        print('wave: ', w)
        print('input spikes: ', input_spiketimes)
        print('li_spike: ', li_spiketimes)
        print('weights_out: ', layer.weights)
        print('capture: ', rvcapture[w])
        print('search: ', rvsearch[w])
        print('backoff: ', rvbackoff[w])
        print('min: ', rvmin[w])
        print('minus: ', rvbackoff[w])
        print('F: ', rvF[w])
        print('')
        
        img_flat = img.flatten()
        img_flat += 1
        
        # generate input spike times
        img_flat[img_flat==float('inf')] = -1
        spike_times = np.array(img_flat)
        reset_times = np.zeros(spike_times.shape)
        for i in range(num_synapse):
            if spike_times[i]>0:
                reset_times[i] = spike_times[i]+(2**wres)-(2**tres)
            else:
                reset_times[i] = -1
                
        # generate output spike time
        li_spiketimes += 1
        li_spiketimes[li_spiketimes==float('inf')] = -1
        li_spiketimes = int(li_spiketimes.item())

        # t-window
        if verbose:
            print('t-window')
        in_bits = ['0'] * num_synapse
        delays = []
        layer_in = [0]
        layer_out = [0]
        delay = 0
        layer_in_string = '0'
        
        for t in range(2**tres):
            if (t in spike_times) or (t == li_spiketimes):
                delays += [delay]
                delay = 0
                time_matches = np.where(spike_times == t)[0]

                # Generate input spike at time t
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '1'

                # Calculate layer_in
                layer_in_string = ''
                for i in range(num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [int(layer_in_string, 2)]
                                
                # Calculate layer_out
                if (t == li_spiketimes):
                    layer_out += [1]
                else:
                    layer_out += [0]

            delay+=1

        layer_in += [int(layer_in_string, 2)]
        delays += [delay]
        li_spiketimes -= (2**tres)
        layer_out += [0]
        
        # w-window
        if verbose:
            print('w-window')
        delay = 0
        for t in range(2**wres-1):
            if (t in reset_times) or (t == li_spiketimes):
                delays += [delay]
                delay = 0
                time_matches = np.where(reset_times == t)[0]
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '0'

                # Calculate layer_in
                layer_in_string = ''
                for i in range(num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [int(layer_in_string, 2)]
                
                # Calculate layer_out
                if (t == li_spiketimes):
                    layer_out += [1]
                else:
                    layer_out += [0]

            delay+=1

        delays+=[delay]
        
        # added for testing
        layer_in+=[0]
        layer_out+=[0]
        delays+=[8]
        delays[0]-=0.5

        if True:
            print('layer_in, layer_out, delays')
            print(layer_in, layer_out, delays)
            print('')
            
        # write brv variables
        f.write('Delay(0.5),\n')
        f.write('capture_brv('+str(int(rvcapture[w].item()))+'),\n')
        f.write('search_brv('+str(int(rvsearch[w].item()))+'),\n')
        f.write('backoff_brv('+str(int(rvbackoff[w].item()))+'),\n')
        f.write('min_brv('+str(int(rvmin[w].item()))+'),\n')
        f.write('minus_brv('+str(int(rvbackoff[w].item()))+'),\n')
        f.write('F_brv('+str(int(''.join(map(str,rvF[w][1:-1].int().tolist())), base=2))+'),\n')
        f.write('\n')
            
        # write input vector and delay
        for i in range(len(layer_in)):
            f.write('layer_in('+str(layer_in[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            if layer_out[i] == 1:
                f.write('EmbeddedCode("""assert (dut_layer_out == 1) else $error(\\"Output error\\"); """),\n')
            f.write('\n')
            
    f.write('layer_in(0),\n')
    f.write('Delay(100),\n')

In [14]:
MNIST_single_column(num_synapse=18, thres=5, tres=1, wres=3, wave=20, verbose=False)

wave:  0
input spikes:  tensor([[0., 0., inf, inf, 0., 0., inf, inf, 0., inf, inf, 0., 0., inf, inf, 0., 0., inf]])
li_spike:  tensor([[[inf]]])
weights_out:  Parameter containing:
tensor([[1., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 0.]])
capture:  tensor(0.)
search:  tensor(1.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0., 0., 0., 0., 1., 0., 0., 0.])

layer_in, layer_out, delays
[0, 209510, 209510, 0] [0, 0, 0, 0] [0.5, 1, 7, 8]

wave:  1
input spikes:  tensor([[inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
li_spike:  tensor([[[inf]]])
weights_out:  Parameter containing:
tensor([[1., 1., 0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 0.]])
capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0., 0., 0., 0., 0., 0., 0., 0.])

layer_in, layer_out, delays
[0, 511, 511, 0] [0, 0, 0, 0] [0.5, 1, 7, 8]

wave:  2
input spikes:  tensor

In [47]:
img, label = mnist_posneg[0]
print(img.flatten())

tensor([0., 0., inf, inf, 0., 0., inf, inf, 0., inf, inf, 0., 0., inf, inf, 0., 0., inf])


In [109]:
# def MNIST_multi_column(num_column=4, num_neuron=1, num_synapse=18, thres=11, tres=1, wres=3, wave=10, verbose=False):
    
#     mnist_posneg = MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor(),PosNeg(0.5),transforms.CenterCrop(6)]))
    
#     # Generate brv random variables
#     rvcapture, rvsearch, rvbackoff, rvmin, rvF = gen_brv(wave=wave, ucapture=512, usearch=512, ubackoff=64, umin=64, wres=wres)
    
#     # Hard-coded PyTorch config
#     layer = TNNColumnLayer(inputsize=6, rfsize=3, stride=3, nprev=2, neurons=num_neuron, theta=thres,\
#     timeres=tres-1, wres=wres, ntype="rnl", ramp=1, w_init="zero", k=1, stoch="low", reward_en=0)
    
#     layer_output = []
#     layer_weights_out = []
    
#     f = open("tnn_mdls/tests/MNIST_multi_column", "w")
    
#     for w in range(wave):
        
#         f.write('# Wave: '+ str(w) +'\n')
#         img, label = mnist_posneg[w]
        
#         output_spiketimes, input_spiketimes, li_spiketimes = layer(img)
#         layer.weights = layer.stdp(input_spiketimes, output_spiketimes, layer.weights,\
#                          rvcapture[w], rvsearch[w], rvbackoff[w], rvmin[w], rvF[w])
        
#         if verbose:
#             print('wave: ', w)
#             print('input spikes: ', input_spiketimes)
#             print('total pixel count: ', len(input_spiketimes.flatten()))
#             print('li_spike: ', li_spiketimes)
#             print('weights_out: ', layer.weights)
#             print('capture: ', rvcapture[w])
#             print('search: ', rvsearch[w])
#             print('backoff: ', rvbackoff[w])
#             print('min: ', rvmin[w])
#             print('minus: ', rvbackoff[w])
#             print('F: ', rvF[w])
#             print('')
        
#         #img_flat = img.flatten()
#         img_flat = (input_spiketimes[0::num_neuron, :]).flatten()
#         img_flat += 1
        
#         # generate input spike times
#         img_flat[img_flat==float('inf')] = -1
#         in_spike_times = np.array(img_flat)
#         reset_times = np.zeros(in_spike_times.shape)
#         for i in range(num_column * num_synapse):
#             if in_spike_times[i]>0:
#                 reset_times[i] = in_spike_times[i]+(2**wres)-(2**tres)
#             else:
#                 reset_times[i] = -1
                
#         # generate output spike time
#         li_spiketimes += 1
#         li_spiketimes[li_spiketimes==float('inf')] = -1
#         out_spike_times = li_spiketimes.flatten()

#         # t-window
#         if verbose:
#             print('t-window')
#         in_bits = ['0'] * num_column * num_synapse
#         out_bits = ['0'] * num_column * num_neuron
#         delays = []
#         layer_in = [0]
#         layer_out = [0]
#         delay = 0
#         layer_in_string = '0'
#         layer_out_string = '0'
        
#         for t in range(2**tres):
#             if (t in in_spike_times) or (t in out_spike_times):
#                 delays += [delay]
#                 delay = 0
#                 in_time_matches = np.where(in_spike_times == t)[0]
#                 out_time_matches = np.where(out_spike_times == t)[0]

#                 # Calculate layer_in vector
#                 for i in range(len(in_time_matches)):
#                     index = in_time_matches[i]
#                     in_bits[index] = '1'

#                 layer_in_string = ''
#                 for i in range(num_column * num_synapse):
#                     layer_in_string += in_bits[i]
#                 layer_in += [hex(int(layer_in_string, 2))]
                                
#                 # Calculate layer_out vector
#                 for i in range(len(out_time_matches)):
#                     index = out_time_matches[i]
#                     out_bits[index] = '1'
                    
#                 layer_out_string = ''
#                 for i in range(num_column * num_neuron):
#                     layer_out_string += out_bits[i]
#                 layer_out += [hex(int(layer_out_string, 2))]

#             delay+=1

#         layer_in += [hex(int(layer_in_string, 2))]
#         delays += [delay]
#         out_spike_times -= (2**tres)
#         layer_out += [0]
        
#         # w-window
#         if verbose:
#             print('w-window')
#         delay = 0
#         for t in range(2**wres-1):
#             if (t in reset_times) or (t in out_spike_times):
#                 delays += [delay]
#                 delay = 0
#                 in_time_matches = np.where(reset_times == t)[0]
#                 out_time_matches = np.where(out_spike_times == t)[0]

#                 # Calculate layer_in vector
#                 for i in range(len(in_time_matches)):
#                     index = in_time_matches[i]
#                     in_bits[index] = '1'

#                 layer_in_string = ''
#                 for i in range(num_column * num_synapse):
#                     layer_in_string += in_bits[i]
#                 layer_in += [hex(int(layer_in_string, 2))]
                                
#                 # Calculate layer_out vector
#                 for i in range(len(out_time_matches)):
#                     index = out_time_matches[i]
#                     out_bits[index] = '1'
                    
#                 layer_out_string = ''
#                 for i in range(num_column * num_neuron):
#                     layer_out_string += out_bits[i]
#                 layer_out += [hex(int(layer_out_string, 2))]

#             delay+=1

#         delays+=[delay]
        
#         # added for testing
#         layer_in+=[0]
#         layer_out+=[0]
#         delays+=[8]
#         delays[0]-=0.5

#         if verbose:
#             print('layer_in, layer_out, delays')
#             print(layer_in, layer_out, delays)
#             print('')
            
#         # write brv variables
#         f.write('Delay(0.5),\n')
#         f.write('capture_brv('+str(int(rvcapture[w].item()))+'),\n')
#         f.write('search_brv('+str(int(rvsearch[w].item()))+'),\n')
#         f.write('backoff_brv('+str(int(rvbackoff[w].item()))+'),\n')
#         f.write('min_brv('+str(int(rvmin[w].item()))+'),\n')
#         f.write('minus_brv('+str(int(rvbackoff[w].item()))+'),\n')
#         f.write('F_brv('+str(int(''.join(map(str,rvF[w][1:-1].int().tolist())), base=2))+'),\n')
#         f.write('\n')
            
#         # write input vector and delay
#         for i in range(len(layer_in)):
#             f.write('layer_in('+str(layer_in[i])+'),\n')
#             f.write('Delay('+str(delays[i])+'),\n')
#             if layer_out[i] != 0:
#                 f.write('EmbeddedCode("""assert (dut_layer_out == '+ layer_out[i] +') else $error(\\"Output error\\"); """),\n')
#             f.write('\n')
            
#     f.write('layer_in(0),\n')
#     f.write('Delay(100),\n')

In [351]:
def MNIST_multi_column(num_column=4, num_neuron=1, num_synapse=18, thres=5, tres=1, wres=3, wave=10, extra_delay=8, verbose=False):
    
    mnist_posneg = MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor(),PosNeg(0.5),transforms.CenterCrop(6)]))
    
    # Generate brv random variables
    rvcapture, rvsearch, rvbackoff, rvmin, rvF = gen_brv(wave=wave, ucapture=64, usearch=64, ubackoff=64, umin=64, wres=wres)
    rvF = torch.tensor([0, 1, 1, 1, 1, 1, 1, 0])
    rvF = rvF.repeat((wave, 1))
    
    # Hard-coded PyTorch config
    layer = TNNColumnLayer(inputsize=6, rfsize=3, stride=3, nprev=2, neurons=num_neuron, theta=thres,\
    timeres=tres-1, wres=wres, ntype="rnl", ramp=1, w_init="zero", k=1, stoch="low", reward_en=0)
    
    layer_output = []
    layer_weights_out = []
    
    f = open("tnn_mdls/tests/MNIST_multi_column", "w")
    f.write('# Init: \n')
    f.write('layer_in(0),\n\n')
    
    for w in range(wave):
        
        f.write('# Wave: '+ str(w) +'\n')
        img, label = mnist_posneg[w]
        
        output_spiketimes, input_spiketimes, li_spiketimes = layer(img)
        layer.weights = layer.stdp(input_spiketimes, output_spiketimes, layer.weights,\
                         rvcapture[w], rvsearch[w], rvbackoff[w], rvmin[w], rvF[w])
        
        if verbose:
            print('wave: ', w)
            print('input spikes: ', input_spiketimes)
#             print('output_spiketimes: ', output_spiketimes)
            print('li_spike: ', li_spiketimes)
            print('weights_out: ', layer.weights)
            print('capture: ', rvcapture[w])
            print('search: ', rvsearch[w])
            print('backoff: ', rvbackoff[w])
            print('min: ', rvmin[w])
            print('minus: ', rvbackoff[w])
            print('F: ', rvF[w], 'F_dim: ', rvF.shape)
            print('')
        
        img_flat = (input_spiketimes[0::num_neuron, :]).flatten()
        img_flat += 1
        
        # generate input spike times
        img_flat[img_flat==float('inf')] = -1
        in_spike_times = np.array(img_flat)
        in_reset_times = np.zeros(in_spike_times.shape)
        for i in range(num_column * num_synapse):
            if in_spike_times[i]>0:
                in_reset_times[i] = in_spike_times[i]+(2**wres)
            else:
                in_reset_times[i] = -1
                
        # generate output spike time
        li_spiketimes += 1
        li_spiketimes[li_spiketimes==float('inf')] = -1
        out_spike_times = li_spiketimes.flatten()
        out_reset_times = np.zeros(out_spike_times.shape)
        for i in range(num_column * num_neuron):
            if out_spike_times[i]>0:
                out_reset_times[i] = out_spike_times[i]+(8)
            else:
                out_reset_times[i] = -1

        in_bits = ['0'] * num_column * num_synapse
        out_bits = ['0'] * num_column * num_neuron
        delays = []
        layer_in = []
        layer_out = []
        delay = 0
        layer_in_string = '0'
        layer_out_string = '0'
        
        for t in range(2**tres + 2**wres - 1 + extra_delay):
            if (t in in_spike_times) or (t in out_spike_times):
                delays += [delay]
                delay = 0
                in_time_matches = np.where(in_spike_times == t)[0]
                out_time_matches = np.where(out_spike_times == t)[0]

                # Calculate layer_in vector
                for i in range(len(in_time_matches)):
                    index = in_time_matches[i]
                    in_bits[index] = '1'

                layer_in_string = ''
                for i in range(num_column * num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [hex(int(layer_in_string, 2))]
                                
                # Calculate layer_out vector
                for i in range(len(out_time_matches)):
                    index = out_time_matches[i]
                    out_bits[index] = '1'
                    
                layer_out_string = ''
                for i in range(num_column * num_neuron):
                    layer_out_string += out_bits[i]
                layer_out += [hex(int(layer_out_string, 2))]
                
            elif (t in in_reset_times) or (t in out_spike_times) or (t in out_reset_times):
                delays += [delay]
                delay = 0
                in_time_matches = np.where(in_reset_times == t)[0]
                out_time_matches = np.where(out_spike_times == t)[0]
                out_reset_matches = np.where(out_reset_times == t)[0]

                # Calculate layer_in vector
                for i in range(len(in_time_matches)):
                    index = in_time_matches[i]
                    in_bits[index] = '0'

                layer_in_string = ''
                for i in range(num_column * num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [hex(int(layer_in_string, 2))]
                                
                # Calculate layer_out vector
                for i in range(len(out_time_matches)):
                    index = out_time_matches[i]
                    out_bits[index] = '1'
                    
                for i in range(len(out_reset_matches)):
                    index = out_reset_matches[i]
                    out_bits[index] = '0'
                    
                layer_out_string = ''
                for i in range(num_column * num_neuron):
                    layer_out_string += out_bits[i]
                layer_out += [hex(int(layer_out_string, 2))]

            delay+=1

        layer_in += [hex(0)]
        layer_out += [hex(0)]
        delays += [delay]
        delays[0]-=0.5

        if verbose:
            print('layer_in, layer_out, delays')
            print(layer_in, layer_out, delays)
            print('')
            
        # write brv variables
        f.write('Delay(0.5),\n')
        f.write('capture_brv('+str(int(rvcapture[w].item()))+'),\n')
        f.write('search_brv('+str(int(rvsearch[w].item()))+'),\n')
        f.write('backoff_brv('+str(int(rvbackoff[w].item()))+'),\n')
        f.write('min_brv('+str(int(rvmin[w].item()))+'),\n')
        f.write('minus_brv('+str(int(rvbackoff[w].item()))+'),\n')
        f.write('F_brv('+str(int(''.join(map(str,rvF[w][1:-1].int().tolist())), base=2))+'),\n')
        f.write('\n')
            
        # write input vector and delay
        layer_in_prev = -1
        layer_out_prev = -1
        for i in range(len(layer_in)):
            f.write('Delay('+str(delays[i])+'),\n')
            if layer_in[i] != layer_in_prev:
                #f.write('layer_in('+str(layer_in[i])+'),\n')
                f.write('EmbeddedCode("""dut_layer_in = ' +'\'h'+ layer_in[i][2:] +';"""),\n')
                layer_in_prev = layer_in[i]
            if layer_out[i] != 0 and layer_out[i] != layer_out_prev and layer_out[i] != '0x0':
                f.write('Delay(0.1),\n')
                f.write('EmbeddedCode("""assert (dut_layer_out == ' +'\'h'+ layer_out[i][2:] +') else $error(\\"Output error: %h\\", dut_layer_out); """),\n')
                delays[i+1] -= 0.1
                layer_out_prev = layer_out[i]
            f.write('\n')
            
    f.write('layer_in(0),\n')
    f.write('Delay(100),\n')

In [352]:
MNIST_multi_column(num_column=4, num_neuron=1, num_synapse=18, thres=5, tres=1, wres=3, wave=500, extra_delay=8, verbose=True)

wave:  0
input spikes:  tensor([[0., 0., 0., inf, 0., 0., inf, inf, 0., inf, inf, inf, 0., inf, inf, 0., 0., inf],
        [inf, inf, inf, inf, inf, inf, 0., 0., inf, 0., 0., 0., 0., 0., 0., inf, inf, 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., inf, 0., 0., inf, inf, inf, inf, inf, inf, 0., inf, inf, 0., 0., 0.]])
li_spike:  tensor([[[inf, inf],
         [inf, inf]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delay

capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0x803fc17f49f307f604', '0x803fc17f49f307f604', '0x0', '0x0', '0x0'] ['0xa', '0xf', '0x5', '0x0', '0x0'] [0.5, 1, 7, 1, 7]

wave:  58
input spikes:  tensor([[inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf],
        [inf, inf, 0., inf, 0., 0., 0., 0., 0., 0., 0., inf, 0., inf, inf, inf, inf, inf],
        [0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf, 0.],
        [0., 0., 0., 0., 0., 0., inf, 0., 0., inf, inf, inf, inf, inf, inf, 0., inf, inf]])
li_spike:  tensor([[[0., 0.],
         [0., 1.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 3., 4., 2., 4., 5., 3., 4., 6.],
        [0., 2., 3., 2., 3., 3., 2., 2., 2., 2., 0., 0., 0., 0., 0., 0., 0., 0.],
        [2., 2., 3., 1., 3., 3., 2., 2., 

input spikes:  tensor([[inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0., inf],
        [inf, 0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf],
        [0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0.],
        [0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0.]])
li_spike:  tensor([[[0., 0.],
         [1., 1.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 6., 0., 5., 7., 4., 7., 7.],
        [0., 1., 4., 1., 4., 5., 3., 3., 4., 3., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 3., 0., 1., 4., 7., 7., 0., 7., 0., 0., 7., 6., 3.],
        [0., 3., 0., 5., 5., 0., 5., 0., 0., 0., 0., 1., 0., 0., 3., 0., 2., 5.]])
capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0xff9ff00

wave:  129
input spikes:  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf],
        [inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf],
        [0., 0., 0., 0., 0., inf, 0., inf, inf, inf, inf, inf, inf, inf, 0., inf, 0., 0.]])
li_spike:  tensor([[[inf, 0.],
         [2., 0.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 5., 0., 5., 7., 4., 7., 7.],
        [0., 1., 5., 2., 5., 5., 4., 4., 4., 3., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 3., 0., 0., 4., 7., 7., 0., 7., 0., 0., 7., 6., 2.],
        [0., 3., 0., 5., 5., 0., 5., 0., 0., 0., 0., 1., 0., 0., 4., 0., 2., 5.]])
capture:  tensor(1.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays

capture:  tensor(0.)
search:  tensor(1.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0x3fe0ff80f803f9236', '0x3fe0ff80f803f9236', '0x3fe0ff80f803f9236', '0x0', '0x0', '0x0', '0x0'] ['0x4', '0xe', '0xf', '0xb', '0x1', '0x0', '0x0'] [0.5, 1, 1, 6, 1, 1, 6]

wave:  158
input spikes:  tensor([[inf, 0., 0., inf, inf, inf, inf, inf, inf, 0., inf, inf, 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., inf, inf, 0., inf, inf, inf, inf, inf, inf, 0., 0., inf, 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
li_spike:  tensor([[[0., 1.],
         [0., 1.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 4., 0., 5., 7., 7., 7., 7.],
        [0., 2., 5., 4., 7., 7., 4., 4., 5., 2., 0., 0., 0., 0., 0., 0., 0

...

weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 4., 7., 3., 7., 7., 7., 7., 7.],
        [0., 2., 3., 5., 7., 7., 5., 6., 7., 4., 1., 3., 0., 0., 2., 0., 0., 2.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 7., 7., 0., 7., 1., 0., 7., 7., 6.],
        [0., 6., 1., 7., 7., 0., 5., 0., 1., 0., 0., 1., 1., 0., 7., 1., 2., 4.]])
capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0x7fc03fe007fdb724', '0x0', '0x0'] ['0xf', '0x0', '0x0'] [0.5, 8, 8]

wave:  259
input spikes:  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf],
        [inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, 0., 0., inf, inf, inf, inf, inf, inf, 0., inf, inf, 0., 0

backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0x4fdbec0925ed3485b', '0x0', '0x0'] ['0xf', '0x0', '0x0'] [0.5, 8, 8]

wave:  291
input spikes:  tensor([[inf, inf, inf, inf, inf, 0., inf, 0., 0., 0., 0., 0., 0., 0., inf, 0., inf, inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., inf, inf, inf, inf, inf, inf, inf, inf, inf],
        [inf, inf, 0., inf, inf, inf, inf, inf, inf, 0., 0., inf, 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., inf, inf, 0., inf, inf, 0., inf, inf, inf, 0., 0., inf, 0., 0., inf]])
li_spike:  tensor([[[0., 0.],
         [0., 0.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 5., 6., 7., 5., 7., 7., 7., 7., 7.],
        [0., 2., 5., 5., 7., 7., 6., 7., 7., 6., 3., 3., 0., 0., 2., 0., 0., 2.],
        [0., 0., 0., 0., 0., 0., 0., 0., 2., 7., 7., 0., 7., 3., 0., 7., 7., 7.],
        [0., 7., 3., 7., 7., 0., 6., 0., 2., 0., 0.

wave:  349
input spikes:  tensor([[inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, 0., inf, inf, 0., 0., 0., 0., 0., 0., inf, 0., 0., inf]])
li_spike:  tensor([[[0., 0.],
         [0., 1.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 6., 7., 7., 6., 7., 7., 7., 7., 7.],
        [0., 1., 5., 5., 7., 7., 7., 7., 7., 7., 7., 6., 0., 0., 5., 0., 0., 3.],
        [0., 0., 0., 0., 0., 0., 0., 0., 2., 7., 7., 0., 7., 6., 0., 7., 7., 7.],
        [0., 7., 6., 7., 7., 0., 7., 0., 5., 0., 0., 0., 7., 0., 7., 4., 3., 5.]])
capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(1.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays


weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 7., 7., 7., 6., 7., 7., 7., 7., 7.],
        [0., 0., 4., 0., 5., 7., 5., 7., 7., 7., 7., 7., 4., 3., 5., 3., 3., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 7., 7., 4., 7., 7., 2., 7., 7., 7.],
        [0., 7., 7., 7., 7., 3., 7., 2., 7., 3., 2., 0., 7., 3., 7., 4., 0., 3.]])
capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0xda12c01ff9236c17f4', '0xda12c01ff9236c17f4', '0x0', '0x0', '0x0'] ['0x7', '0xf', '0x8', '0x0', '0x0'] [0.5, 1, 7, 1, 7]

wave:  402
input spikes:  tensor([[0., inf, inf, 0., inf, inf, inf, inf, inf, inf, 0., 0., inf, 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., inf, 0., 0., 0., 0., inf, inf, inf, inf, 0., inf, inf, inf, inf, 0., 0., 0.],
        [0., 0., 0

capture:  tensor(0.)
search:  tensor(0.)
backoff:  tensor(0.)
min:  tensor(0.)
minus:  tensor(0.)
F:  tensor([0, 1, 1, 1, 1, 1, 1, 0]) F_dim:  torch.Size([500, 8])

layer_in, layer_out, delays
['0x7fc01ff7fc03f604', '0x7fc01ff7fc03f604', '0x0', '0x0', '0x0'] ['0xd', '0xf', '0x2', '0x0', '0x0'] [0.5, 4, 4, 4, 4]

wave:  458
input spikes:  tensor([[inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [inf, inf, inf, inf, inf, inf, inf, inf, inf, 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
li_spike:  tensor([[[0., 0.],
         [0., 0.]]])
weights_out:  Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 7., 7., 7., 7., 7., 7., 7., 7., 7.],
        [0., 0., 0., 0., 4., 7., 7., 7., 7., 7., 7., 7., 4., 3., 7., 2., 2., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [38]:
def simple_easy(num_synapse=1, thres=3, tres=1, wres=3, wave=10, verbose=False):
    
    # Generate brv random variables
    rvcapture, rvsearch, rvbackoff, rvmin, rvF = gen_brv(wave=wave, ucapture=0, usearch=1024, ubackoff=0, umin=0, wres=3)
    
    f = open("tnn_mdls/tests/simple_easy", "w")
    
    for w in range(wave):
        
        f.write('# Wave: '+ str(w) +'\n')

        img_flat = np.array([1, 1])
        
        # generate input spike times
        spike_times = np.array(img_flat)
        reset_times = np.zeros(spike_times.shape)
        for i in range(num_synapse):
            if spike_times[i]>0:
                reset_times[i] = spike_times[i]+(2**wres)-(2**tres)
            else:
                reset_times[i] = -1

        # t-window
        if verbose:
            print('t-window')
        in_bits = ['0'] * num_synapse
        delays = []
        layer_in = [0]
        layer_out = [0]
        delay = 0
        layer_in_string = '0'
        
        for t in range(2**tres):
            if (t in spike_times):
                delays += [delay]
                delay = 0
                time_matches = np.where(spike_times == t)[0]

                # Generate input spike at time t
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '1'

                # Calculate layer_in
                layer_in_string = ''
                for i in range(num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [int(layer_in_string, 2)]

            delay+=1

        layer_in += [int(layer_in_string, 2)]
        delays += [delay]
        
        # w-window
        if verbose:
            print('w-window')
        delay = 0
        for t in range(2**wres-1):
            if (t in reset_times):
                delays += [delay]
                delay = 0
                time_matches = np.where(reset_times == t)[0]
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '0'

                # Calculate layer_in
                layer_in_string = ''
                for i in range(num_synapse):
                    layer_in_string += in_bits[i]
                layer_in += [int(layer_in_string, 2)]

            delay+=1

        delays+=[delay]
        
        # added for testing
        layer_in+=[0]
        delays+=[8]

        if True:
            print('layer_in, delays')
            print(layer_in, delays)
            print('')
            
        # write brv variables
        f.write('capture_brv('+str(int(rvcapture[w].item()))+'),\n')
        f.write('search_brv('+str(int(rvsearch[w].item()))+'),\n')
        f.write('backoff_brv('+str(int(rvbackoff[w].item()))+'),\n')
        f.write('min_brv('+str(int(rvmin[w].item()))+'),\n')
        f.write('minus_brv('+str(int(rvbackoff[w].item()))+'),\n')
        f.write('F_brv('+str(int(''.join(map(str,rvF[w][1:-1].int().tolist())), base=2))+'),\n')
        f.write('\n')
            
        # write input vector and delay
        for i in range(len(layer_in)):
            f.write('layer_in('+str(layer_in[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')
            
    f.write('layer_in(0),\n')
    f.write('Delay(100),\n')

In [39]:
simple_easy(num_synapse=2, thres=3, tres=1, wres=3, wave=10, verbose=False)

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

layer_in, delays
[0, 3, 3, 0] [1, 1, 7, 8]

